# Task
Analyze Oracle SQL view scripts from the BigQuery table "tnn-consumer-common-nx.temp.ccm_user_views" using the `gemini-2.5-flash` model on Vertex AI (project ID: 'tnn-pnx-consumer-common-ai', location: 'europe-west1') to extract their functionality and data sources, then save the structured analysis to a JSON file.

In [51]:
import subprocess
import sys
import os

print("="*60)
print("GCP AUTHENTICATION CHECK")
print("="*60)

try:
    # Check if gcloud credentials are available
    print("\nChecking GCP login status...")
    
    # Try to get the current GCP project
    result = subprocess.run(
        ['gcloud', 'config', 'get-value', 'project'],
        capture_output=True,
        text=True,
        timeout=10
    )
    
    if result.returncode == 0 and result.stdout.strip():
        current_project = result.stdout.strip()
        print(f"✓ Currently logged in to GCP")
        print(f"  Active project: {current_project}")
        
        # Also check if application-default credentials exist
        print("\nVerifying application-default credentials...")
        result = subprocess.run(
            ['gcloud', 'auth', 'application-default', 'print-access-token'],
            capture_output=True,
            text=True,
            timeout=10
        )
        
        if result.returncode == 0:
            print("✓ Application-default credentials valid")
            print("\nReady to run analysis cells.\n")
        else:
            raise Exception("Application-default credentials not set")
    else:
        raise Exception("Not logged in to GCP")
        
except Exception as e:
    print(f"\n✗ GCP authentication failed!")
    print(f"Error: {str(e)}\n")
    print("="*60)
    print("RUNNING LOGIN COMMANDS...")
    print("="*60)
    
    try:
        print("\n[1/2] Running: gcloud auth login")
        print("(A browser window will open. Log in and authorize.)\n")
        result1 = subprocess.run(
            ['gcloud', 'auth', 'login'],
            timeout=300
        )
        
        if result1.returncode == 0:
            print("\n✓ gcloud auth login completed")
        else:
            print("\n✗ gcloud auth login failed")
            sys.exit(1)
        
        print("\n[2/2] Running: gcloud auth application-default login")
        print("(A browser window will open. Log in and authorize.)\n")
        result2 = subprocess.run(
            ['gcloud', 'auth', 'application-default', 'login'],
            timeout=300
        )
        
        if result2.returncode == 0:
            print("\n✓ gcloud auth application-default login completed")
            print("\nAuthentication successful! Ready to run analysis cells.\n")
        else:
            print("\n✗ gcloud auth application-default login failed")
            sys.exit(1)
            
    except subprocess.TimeoutExpired:
        print("\n✗ Authentication timed out")
        sys.exit(1)
    except Exception as e:
        print(f"\n✗ Error during login: {str(e)}")
        sys.exit(1)

print("="*60)


GCP AUTHENTICATION CHECK

Checking GCP login status...
✓ Currently logged in to GCP
  Active project: tnn-consumer-common-nx

Verifying application-default credentials...

✗ GCP authentication failed!
Error: Application-default credentials not set

RUNNING LOGIN COMMANDS...

[1/2] Running: gcloud auth login
(A browser window will open. Log in and authorize.)



Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=aO85sAPVBQFgQNNhNAMiN8qsAQN2Hq&access_type=offline&code_challenge=piQ_YzsIzhRqD1efCcECbMwO6ff4GdKlJIFPCngpTn8&code_challenge_method=S256


You are now logged in as [ornulf.thomassen@telenor.no].
Your current project is [tnn-consumer-common-nx].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID



✓ gcloud auth login completed

[2/2] Running: gcloud auth application-default login
(A browser window will open. Log in and authorize.)



Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=dxZeZCME4KjtTnAtEL3geEITDpfRYP&access_type=offline&code_challenge=P8Jfy8sNSlNkVcylZ91mU08j1p71X1wPlRRGavP-G-c&code_challenge_method=S256


Credentials saved to file: [/Users/t736135/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).



✓ gcloud auth application-default login completed

Authentication successful! Ready to run analysis cells.



Cannot add the project "tnn-consumer-common-nx" to ADC as the quota project because the account in ADC does not have the "serviceusage.services.use" permission on this project. You might receive a "quota_exceeded" or "API not enabled" error. Run $ gcloud auth application-default set-quota-project to add a quota project.


In [52]:
import json
import subprocess
import tqdm
import pandas as pd
import vertexai
from vertexai.preview.generative_models import GenerativeModel

# Initialize Vertex AI
project_id = 'tnn-pnx-consumer-common-ai'
location = 'europe-west1'
vertexai.init(project=project_id, location=location)
model_gemini = GenerativeModel('gemini-2.5-flash')

# BigQuery project where data is stored
bigquery_project = 'tnn-consumer-common-nx'

# Ensure prompt_template is defined (from previous steps)
prompt_template = """
Analyze the following Oracle SQL view script and extract its main functionality/purpose and all data sources (tables or views) it directly queries.
Provide the output in a JSON format with two keys: "functionality" (string) and "data_sources" (list of strings).
Example output:
{{
  "functionality": "Calculates daily sales by product category.",
  "data_sources": ["SALES_TABLE", "PRODUCT_CATEGORIES"]
}}

SQL Script:
---
{sql_script}
---
"""

# Ensure bigquery_table_ids is defined (from previous steps)
bigquery_table_ids = [
    'tnn-consumer-common-nx.temp.ccm_user_views',
    'tnn-consumer-common-nx.temp.clm_adm_user_views',
    'tnn-consumer-common-nx.temp.clm_ccm_user_views',
    'tnn-consumer-common-nx.temp.crm_analyse_user_views'
]

for table_id in bigquery_table_ids:
    print(f"\nProcessing table (DATA PREP): {table_id}")

    # 2. Construct a SQL query to select all columns from the current table_id
    query = f"""SELECT * FROM `{table_id}`"""

    # 3. Execute the SQL query using bq CLI and load the results into a pandas DataFrame
    try:
        cmd = [
            'bq', 'query',
            f'--project_id={bigquery_project}',
            '--use_legacy_sql=false',
            '--format=json',
            '--max_rows=10000',
            query
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        
        if result.returncode == 0:
            data = json.loads(result.stdout)
            df_current_views = pd.DataFrame(data)
            print(f"Loaded {len(df_current_views)} views from {table_id}.")
        else:
            print(f"Error loading data from {table_id}: {result.stderr}")
            continue
            
    except Exception as e:
        print(f"Error loading data from {table_id}: {e}")
        continue

    # 4. Save raw data to CSV for LLM processing
    prep_filename = f'{table_id.replace(".", "_").replace("/", "_")}_views_preprocessed.csv'
    df_current_views.to_csv(prep_filename, index=False)
    print(f"Preprocessed views saved to {prep_filename}")

print("\nViews data preparation complete!")

/opt/homebrew/lib/python3.11/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()



Processing table (DATA PREP): tnn-consumer-common-nx.temp.ccm_user_views
Loaded 512 views from tnn-consumer-common-nx.temp.ccm_user_views.
Preprocessed views saved to tnn-consumer-common-nx_temp_ccm_user_views_views_preprocessed.csv

Processing table (DATA PREP): tnn-consumer-common-nx.temp.clm_adm_user_views
Loaded 81 views from tnn-consumer-common-nx.temp.clm_adm_user_views.
Preprocessed views saved to tnn-consumer-common-nx_temp_clm_adm_user_views_views_preprocessed.csv

Processing table (DATA PREP): tnn-consumer-common-nx.temp.clm_ccm_user_views
Loaded 253 views from tnn-consumer-common-nx.temp.clm_ccm_user_views.
Preprocessed views saved to tnn-consumer-common-nx_temp_clm_ccm_user_views_views_preprocessed.csv

Processing table (DATA PREP): tnn-consumer-common-nx.temp.crm_analyse_user_views
Loaded 253 views from tnn-consumer-common-nx.temp.crm_analyse_user_views.
Preprocessed views saved to tnn-consumer-common-nx_temp_crm_analyse_user_views_views_preprocessed.csv

Views data prepa

In [ ]:
import json
import pandas as pd
import tqdm
import os
import time
import vertexai
from vertexai.preview.generative_models import GenerativeModel

# Initialize Vertex AI
project_id = 'tnn-pnx-consumer-common-ai'
location = 'europe-west1'
vertexai.init(project=project_id, location=location)
model_gemini = GenerativeModel('gemini-2.5-flash')

# The prompt_template is already defined in the previous step.
# The bigquery_table_ids is already defined in the previous step.

for table_id in bigquery_table_ids:
    print(f"\n{'='*60}")
    print(f"Processing table (LLM ANALYSIS): {table_id}")
    print(f"{'='*60}")

    # 1. Read the preprocessed CSV file
    prep_filename = f'{table_id.replace(".", "_").replace("/", "_")}_views_preprocessed.csv'
    final_output_filename = f'{table_id.replace(".", "_").replace("/", "_")}_views_analysis.json'
    checkpoint_filename = f'{table_id.replace(".", "_").replace("/", "_")}_views_checkpoint.json'
    
    if not os.path.exists(prep_filename):
        print(f"Preprocessed file {prep_filename} not found. Skipping this table.")
        continue
    
    df_current_views = pd.read_csv(prep_filename)
    print(f"Loaded {len(df_current_views)} views from {prep_filename}")

    # 2. Load checkpoint (previously completed views) if it exists
    completed_views = set()
    current_analysis_results = []
    
    if os.path.exists(checkpoint_filename):
        with open(checkpoint_filename, 'r') as f:
            checkpoint_data = json.load(f)
            current_analysis_results = checkpoint_data.get('results', [])
            completed_views = set(checkpoint_data.get('completed', []))
            print(f"Loaded checkpoint: {len(completed_views)} views already analyzed")
    
    # 3. Iterate through views, skipping already-completed ones
    print(f"Analyzing {len(df_current_views) - len(completed_views)} remaining views...")
    
    run_start = time.perf_counter()
    checkpoint_interval = 5  # Save checkpoint every N views
    
    for idx, row in enumerate(tqdm.tqdm(df_current_views.iterrows(), total=df_current_views.shape[0], desc=f"Views in {table_id}")):
        view_name = row[1]['view_name']
        
        # Skip if already completed
        if view_name in completed_views:
            continue
        
        sql_script = row[1]['text_long'] if pd.notna(row[1]['text_long']) else row[1]['text_vc']
        
        # Format and send prompt
        full_prompt = prompt_template.format(sql_script=sql_script)
        
        iter_start = time.perf_counter()
        try:
            response = model_gemini.generate_content(full_prompt)
            response_text = response.text.strip()

            # Parse response
            if response_text.startswith("```json") and response_text.endswith("```"):
                response_text = response_text[7:-3].strip()
            elif response_text.startswith("```") and response_text.endswith("```"):
                response_text = response_text[3:-3].strip()

            parsed_response = json.loads(response_text)
            functionality = parsed_response.get('functionality', 'N/A')
            data_sources = parsed_response.get('data_sources', [])

            current_analysis_results.append({
                'view_name': view_name,
                'functionality': functionality,
                'data_sources': data_sources
            })
            
            completed_views.add(view_name)
            
        except Exception as e:
            current_analysis_results.append({
                'view_name': view_name,
                'functionality': 'Error during analysis',
                'data_sources': [],
                'error': str(e)
            })
            completed_views.add(view_name)
        
        iter_elapsed = time.perf_counter() - iter_start
        
        # Save checkpoint periodically
        if len(completed_views) % checkpoint_interval == 0:
            checkpoint_data = {
                'completed': list(completed_views),
                'results': current_analysis_results
            }
            with open(checkpoint_filename, 'w') as f:
                json.dump(checkpoint_data, f, indent=2)
            
            elapsed = time.perf_counter() - run_start
            avg_time = elapsed / len(completed_views)
            print(f"[CHECKPOINT] {len(completed_views)}/{len(df_current_views)} analyzed (avg {avg_time:.1f}s each)")

    # 4. Convert results to DataFrame and save final JSON
    df_current_analysis = pd.DataFrame(current_analysis_results)
    df_current_analysis.to_json(final_output_filename, orient='records', indent=4)
    print(f"\nFinal analysis saved to {final_output_filename}")
    
    # 5. Clean up checkpoint file after successful completion
    if os.path.exists(checkpoint_filename):
        os.remove(checkpoint_filename)
        print(f"Checkpoint cleaned up")

print("\n" + "="*60)
print("All views processed!")
print("="*60)


In [1]:
bigquery_table_ids = ["tnn-consumer-common-nx.temp.clm_adm_user_source"]
print(f"BigQuery table ID defined: {bigquery_table_ids}")

BigQuery table ID defined: ['tnn-consumer-common-nx.temp.clm_adm_user_source']


In [2]:
prompt_template = """
Analyze the following Oracle SQL procedure script and extract its main functionality/purpose, all data sources (tables or views) it directly queries, and all tables it creates (identifying if they are temporary).
Provide the output in a JSON format with three keys: "functionality" (string), "data_sources" (list of strings), and "target_tables" (list of objects).
Each object in "target_tables" should have "table_name" (string) and "is_temporary" (boolean or null if temporariness cannot be determined).

Example output:
{{
  "functionality": "Processes daily sales data, inserting aggregated results into a permanent sales summary table and using a temporary table for intermediate calculations.",
  "data_sources": ["SALES_TRANSACTIONS", "PRODUCTS", "CUSTOMERS"],
  "target_tables": [
    {{
      "table_name": "DAILY_SALES_SUMMARY",
      "is_temporary": false
    }},
    {{
      "table_name": "TEMP_SALES_AGG",
      "is_temporary": true
    }}
  ]
}}

SQL Script:
---
{sql_script}
---
"""

print("Prompt template for procedure analysis defined.")

Prompt template for procedure analysis defined.


In [12]:
import json
import tqdm
import pandas as pd
import subprocess
import vertexai
from vertexai.preview.generative_models import GenerativeModel

# Ensure client and model_gemini are initialized (from previous steps)
# BigQuery project where data is stored
bigquery_project = 'tnn-consumer-common-nx'

project_id = 'tnn-pnx-consumer-common-ai'
location = 'europe-west1'
vertexai.init(project=project_id, location=location)
model_gemini = GenerativeModel('gemini-2.5-flash')

# The prompt_template is already defined in the previous step.
# The bigquery_table_ids is already defined in the previous step as ['tnn-consumer-common-nx.temp.ccm_user_source']

for table_id in bigquery_table_ids:
    print(f"\nProcessing table: {table_id}")

    # 2. Construct a SQL query to select all columns from the current table_id
    query = f"""SELECT * FROM `{table_id}`"""

    # 3. Execute the SQL query and load the results into a pandas DataFrame
    try:
        cmd = [
            'bq', 'query',
            f'--project_id={bigquery_project}',
            '--use_legacy_sql=false',
            '--format=json',
            '--max_rows=100000', # Increased max_rows for procedures
            query
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            data = json.loads(result.stdout)
            df_current_procedures = pd.DataFrame(data)
            print(f"Loaded {len(df_current_procedures)} rows from {table_id}.")
        else:
            print(f"Error loading data from {table_id}: {result.stderr}")
            continue
    except Exception as e:
        print(f"Error loading data from {table_id}: {e}")
        continue

    # Preprocess: Group by procedure name, sort by line number, and concatenate the text lines
    # Assuming 'name' is the procedure name column and 'line' is the line number column
    df_procedures_scripts = df_current_procedures.sort_values(by=['name', 'line']).groupby('name')['text'].apply(lambda x: '\n'.join(x)).reset_index()
    df_procedures_scripts.rename(columns={'name': 'procedure_name', 'text': 'sql_script'}, inplace=True)

    # 4. Initialize an empty list to store the analysis outcomes
    current_analysis_results = []

    # 5. Iterate through each row of df_procedures_scripts (which now contains full scripts)
    for index, row in tqdm.tqdm(df_procedures_scripts.iterrows(), total=df_procedures_scripts.shape[0], desc=f"Analyzing procedures in {table_id}"):
        procedure_name = row['procedure_name'] # Get procedure name from the new DataFrame
        sql_script = row['sql_script'] # Get the full script from the new DataFrame

        # 7. Format the prompt_template with the extracted sql_script
        full_prompt = prompt_template.format(sql_script=sql_script)

        try:
            # 8. Send the formatted prompt to model_gemini.generate_content()
            response = model_gemini.generate_content(full_prompt)
            response_text = response.text.strip()

            # 9. Parse the LLM's response text, cleaning up any markdown formatting
            if response_text.startswith("```json") and response_text.endswith("```"):
                response_text = response_text[7:-3].strip()
            elif response_text.startswith("```") and response_text.endswith("```"):
                response_text = response_text[3:-3].strip()

            # 10. Load the JSON string into a Python dictionary and extract functionality, data_sources, and target_tables
            parsed_response = json.loads(response_text)
            functionality = parsed_response.get('functionality', 'N/A')
            data_sources = parsed_response.get('data_sources', [])
            target_tables = parsed_response.get('target_tables', [])

            # 11. Append a dictionary containing procedure_name, functionality, data_sources, and target_tables
            current_analysis_results.append({
                'procedure_name': procedure_name,
                'functionality': functionality,
                'data_sources': data_sources,
                'target_tables': target_tables
            })
        except Exception as e:
            # 12. Implement robust error handling
            current_analysis_results.append({
                'procedure_name': procedure_name,
                'functionality': 'Error during analysis',
                'data_sources': [],
                'target_tables': [],
                'error': str(e)
            })

    # 13. Convert current_analysis_results into a pandas DataFrame
    df_current_analysis = pd.DataFrame(current_analysis_results)

    # 14. Generate a unique output filename as specified in the task
    output_filename = f'{table_id.replace(".", "_").replace("/", "_")}_procedures_analysis.json'

    # 15. Save df_current_analysis to the generated JSON filename
    df_current_analysis.to_json(output_filename, orient='records', indent=4)

    # 16. Print a confirmation message
    print(f"Analysis results for {table_id} saved to {output_filename}")

print("\nAll BigQuery tables processed.")


Processing table: tnn-consumer-common-nx.temp.clm_adm_user_source
Loaded 100000 rows from tnn-consumer-common-nx.temp.clm_adm_user_source.


Analyzing procedures in tnn-consumer-common-nx.temp.clm_adm_user_source:   0%|          | 0/235 [00:00<?, ?it/s]/opt/homebrew/lib/python3.11/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
Analyzing procedures in tnn-consumer-common-nx.temp.clm_adm_user_source: 100%|██████████| 235/235 [1:20:08<00:00, 20.46s/it] 

Analysis results for tnn-consumer-common-nx.temp.clm_adm_user_source saved to tnn-consumer-common-nx_temp_clm_adm_user_source_procedures_analysis.json

All BigQuery tables processed.
